#  Regresión con criterio: Lasso y variables de ingeniero

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** resistencia a compresión del hormigón (1030 mezclas ensayadas · dataset de Yeh, UCI Machine Learning Repository)

> 🎯 **Mensaje clave:** qué variables meto y en qué forma: el criterio de ingeniero propone y la validación cruzada decide.

> ✏️ **VERSIÓN ALUMNO.** Rellena los huecos `___` y comenta las preguntas en clase

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Punto de partida: la tabla de modelos |
| 2 | Lasso sobre el polinomio de grado 3 |
| 3 | Variables con criterio de ingeniero: a/c y log(edad) |
| 4 | ¿Qué modelo llevamos a la planta? |

### Imports y configuración básica

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # avisos del Lasso con muchas columnas

# Colores que usaremos en todo el bloque
AZUL = "#4C72B0"      # datos de entrenamiento
NARANJA = "#DD8452"   # datos de prueba / validación
ROJO = "#C44E52"      # modelo

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)


def rmse(y_real, y_pred):
    """Raíz del error cuadrático medio, en las unidades de la respuesta (MPa)."""
    return np.sqrt(mean_squared_error(y_real, y_pred))

---
## 1. Punto de partida: la tabla de modelos

Mismo fichero, **mismo reparto** (20 % de prueba, que no tocamos) y **mismos 5 pliegues** de validación cruzada que hasta ahora. Rehacemos la tabla con los tres modelos de referencia.

In [2]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
ruta = "/lustre/geocean/WORK/users/alonsoap/personal/dodencia/AnaliticaDeDatos/datos/clases/clase_1_analisis_exploratorio_de_datos/Concrete_Data.xls"
concrete_data = pd.read_excel(ruta, index_col=False)

concrete_data.columns = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
                         "arido_grueso", "arido_fino", "edad", "resistencia"]

variables = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
             "arido_grueso", "arido_fino", "edad"]
X = concrete_data[variables]
y = concrete_data["resistencia"]

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)   # mismos pliegues para todo el notebook

print("Entrenamiento:", len(X_ent), "mezclas · Prueba:", len(X_pru), "mezclas (no las tocamos)")

Entrenamiento: 824 mezclas · Prueba: 206 mezclas (no las tocamos)


In [3]:
tabla_modelos = []


def evaluar_modelo(nombre, modelo, X, y, comentario=""):
    """Validación cruzada de 5 pliegues (siempre los mismos) y una fila nueva en la tabla de modelos."""
    res = cross_validate(modelo, X, y, cv=cv,
                         scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])
    fila = {"Modelo": nombre,
            "MAE (MPa)": round(-res["test_neg_mean_absolute_error"].mean(), 2),
            "RMSE (MPa)": round(-res["test_neg_root_mean_squared_error"].mean(), 2),
            "R²": round(res["test_r2"].mean(), 3),
            "Comentario": comentario}
    tabla_modelos.append(fila)
    return fila


def modelo_polinomico(grado):
    """Polinomio de grado `grado` con las 8 variables (escalamos antes de elevar a potencias)."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(grado), LinearRegression())


def knn_escalado(k):
    return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))

✏️ **Rellena el hueco:** el polinomio que mejor funcionaba con las 8 variables era el de **grado 3**.

In [4]:
evaluar_modelo("Regresión lineal", LinearRegression(), X_ent, y_ent, "8 variables · rígida")
evaluar_modelo("Polinomio de grado 3", modelo_polinomico(___), X_ent, y_ent, "164 columnas")   # ✏️
evaluar_modelo("KNN escalado (K = 3)", knn_escalado(3), X_ent, y_ent, "K elegido por VC")

pd.DataFrame(tabla_modelos)

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/nfs/home/geocean/alonsoap/miniforge3/envs/bluemath_tk/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/nfs/home/geocean/alonsoap/miniforge3/envs/bluemath_tk/lib/python3.12/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nfs/home/geocean/alonsoap/miniforge3/envs/bluemath_tk/lib/python3.12/site-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^^^^

El RMSE medio esconde algo. Miramos el error **en cada pliegue**:

In [ ]:
for nombre, modelo in [("Regresión lineal", LinearRegression()),
                       ("Polinomio de grado 3", modelo_polinomico(3))]:
    e = -cross_val_score(modelo, X_ent, y_ent, cv=cv, scoring="neg_root_mean_squared_error")
    print(f"{nombre:22s} → RMSE por pliegue: {np.round(e, 1)}  (media {e.mean():.1f} ± {e.std():.1f} MPa)")

> ### 💡 Preguntas
>
> 1) ¿Qué modelo es hoy el mejor de la tabla? ¿Cuántas columnas usa para 824 mezclas?
>
> 2) ¿Qué le pasa al error del polinomio de un pliegue a otro? ¿Qué nos dice eso del modelo?

---
## 2. Lasso sobre el polinomio de grado 3

El Lasso ajusta por mínimos cuadrados, pero **multa** cada coeficiente distinto de cero:

$$\min_{\beta}\; \underbrace{\sum_{i=1}^{n}\left(y_i - \hat y_i\right)^2}_{\text{RSS: ajuste}} \;+\; \underbrace{\lambda \sum_{j=1}^{p} |\beta_j|}_{\text{multa}}$$

- $\lambda = 0$ → mínimos cuadrados, el polinomio de siempre.
- $\lambda$ grande → todos los $\beta_j = 0$ (predice la media).
- En medio → algunos coeficientes valen **exactamente cero**: el Lasso elige qué columnas se quedan.

En scikit-learn $\lambda$ se llama **`alpha`**. El pipeline es: escalar → crear las columnas del polinomio → **volver a escalar** (para que la multa no dependa de las unidades de cada columna) → Lasso.

### 2.1. Mover el mando a mano

✏️ **Rellena el hueco:** una columna «sobrevive» si su coeficiente es **distinto de cero** (`!=`).

In [ ]:
def polinomio_lasso(alpha):
    return make_pipeline(StandardScaler(), PolynomialFeatures(3, include_bias=False),
                         StandardScaler(), Lasso(alpha=alpha, max_iter=50000))


alphas = [0.001, 0.01, 0.1, 1, 10]
n_columnas, err_cv = [], []
for a in alphas:
    modelo = polinomio_lasso(a).fit(X_ent, y_ent)
    n_columnas.append(int(np.sum(modelo[-1].coef_ ___ 0)))   # ✏️
    err_cv.append(-cross_val_score(polinomio_lasso(a), X_ent, y_ent, cv=cv,
                                   scoring="neg_root_mean_squared_error").mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(alphas, n_columnas, marker="o", color=AZUL, linewidth=2)
axes[0].set_ylabel("Columnas distintas de cero (de 164)")
axes[1].plot(alphas, err_cv, marker="o", color=NARANJA, linewidth=2)
axes[1].set_ylabel("RMSE de validación cruzada (MPa)")
for ax in axes:
    ax.set_xscale("log"); ax.set_xlabel("alpha (λ) · más multa →"); estilo(ax)
plt.tight_layout()
plt.show()

pd.DataFrame({"alpha": alphas, "Columnas ≠ 0": n_columnas, "RMSE VC (MPa)": np.round(err_cv, 2)})

> ### 💡 Preguntas
>
> 1) ¿Qué pasa con el número de columnas al subir `alpha`? ¿Y con el error?
>
> 2) ¿Dónde hay sobreajuste y dónde infraajuste en la gráfica de la derecha?

### 2.2. Elegir `alpha` por validación cruzada

`LassoCV` prueba una rejilla de valores de `alpha` con validación cruzada y se queda con el mejor.

✏️ **Rellena el hueco:** usa los **mismos pliegues** (`cv`) de todo el notebook.

In [ ]:
polinomio_lasso_cv = make_pipeline(StandardScaler(), PolynomialFeatures(3, include_bias=False),
                                   StandardScaler(), LassoCV(cv=___, max_iter=50000))   # ✏️

evaluar_modelo("Polinomio de grado 3 + Lasso", polinomio_lasso_cv, X_ent, y_ent, "alpha elegido por VC")

polinomio_lasso_cv.fit(X_ent, y_ent)
print(f"alpha elegido: {polinomio_lasso_cv[-1].alpha_:.4f}")
print(f"Columnas distintas de cero: {np.sum(polinomio_lasso_cv[-1].coef_ != 0)} de 164")

e = -cross_val_score(polinomio_lasso_cv, X_ent, y_ent, cv=cv, scoring="neg_root_mean_squared_error")
print(f"RMSE por pliegue: {np.round(e, 1)}  (media {e.mean():.1f} ± {e.std():.1f} MPa)")

> ### 💡 Preguntas
>
> 1) Compara con el polinomio sin Lasso: ¿cuánto baja el error medio? ¿Y la variación entre pliegues?
>
> 2) ¿Cuántas columnas ha dejado a cero el Lasso?

---
## 3. Variables con criterio de ingeniero

En lugar de 164 columnas, le damos a la **regresión lineal** dos columnas con sentido físico:

- **Relación agua/cemento** (ley de Abrams): $\;a/c = \dfrac{\text{agua}}{\text{cemento}}$. Más a/c → más porosidad → menos resistencia.
- **Logaritmo de la edad**: el hormigón endurece deprisa al principio y cada vez más despacio. En logaritmo, esa curva es casi una recta. Sustituye a la edad en días.

El modelo sigue siendo lineal: solo cambian las columnas. Cada columna nueva se calcula con datos de la propia mezcla, así que no hay fuga de datos.

✏️ **Rellena el hueco:** el logaritmo en numpy es `np.log`.

In [ ]:
X_ing = X.copy()
X_ing["agua_cemento"] = X["agua"] / X["cemento"]
X_ing["log_edad"] = ___(X["edad"])   # ✏️
X_ing = X_ing.drop(columns="edad")

X_ing_ent = X_ing.loc[X_ent.index]      # las mismas mezclas de entrenamiento

evaluar_modelo("Lineal + a/c + log(edad)", LinearRegression(), X_ing_ent, y_ent, "9 variables con sentido físico")
pd.DataFrame(tabla_modelos)

> ### 💡 Preguntas
>
> 1) ¿Acertasteis la apuesta? ¿Cuánto mejora la lineal con dos columnas nuevas?
>
> 2) ¿Cuánto le falta para llegar al polinomio con Lasso? ¿Con cuántas columnas cada uno?

---
## 4. ¿Qué modelo llevamos a la planta?

La tabla mide el error **dentro del rango de los datos**. Comparamos los dos modelos nuevos en dos cosas que la tabla no mide.

### 4.1. ¿Se puede leer el modelo?

In [ ]:
lineal_ing = LinearRegression().fit(X_ing_ent, y_ent)
display(pd.DataFrame({"Coeficiente": lineal_ing.coef_.round(3)}, index=X_ing.columns))

nombres = polinomio_lasso_cv[1].get_feature_names_out(variables)
coef = polinomio_lasso_cv[-1].coef_
orden = np.argsort(-np.abs(coef))[:8]
print("Polinomio + Lasso: las 8 columnas con más peso (de",
      np.sum(coef != 0), "distintas de cero)")
pd.DataFrame({"Columna": nombres[orden], "Coeficiente": coef[orden].round(2)})

> ### 💡 Preguntas
>
> 1) En la lineal, ¿qué signo tienen a/c y log(edad)? ¿Tiene sentido físico?
>
> 2) ¿Sabríais explicar a una dirección facultativa qué significa `superplastificante edad^2`?

### 4.2. ¿Extrapola?

Entrenamos **solo con los ensayos de hasta 90 días** y predecimos los de **más de 90 días** (fuera del rango de entrenamiento).

✏️ **Rellena el hueco:** el límite de edad es **90** días.

In [ ]:
jovenes = X["edad"] <= ___   # ✏️
viejas = ~jovenes

resultados = []
for nombre, modelo, Xm in [("Polinomio de grado 3", modelo_polinomico(3), X),
                           ("Polinomio de grado 3 + Lasso", polinomio_lasso_cv, X),
                           ("Lineal + a/c + log(edad)", LinearRegression(), X_ing)]:
    modelo.fit(Xm[jovenes], y[jovenes])
    pred = modelo.predict(Xm[viejas])
    resultados.append({"Modelo": nombre, "RMSE (MPa)": round(rmse(y[viejas], pred), 1),
                       "Predicción máxima (MPa)": round(pred.max())})

print(f"{viejas.sum()} ensayos de más de 90 días · resistencia máxima observada: {y[viejas].max():.0f} MPa")
pd.DataFrame(resultados)

> ### 💡 Preguntas
>
> 1) ¿Qué modelo se dispara? ¿Lo arregla el Lasso?
>
> 2) ¿Por qué la lineal con log(edad) se comporta mucho mejor fuera del rango?
>
> 3) ¿Nos habría avisado de esto la validación cruzada?

### La tabla de modelos, completa

In [ ]:
tabla = pd.DataFrame(tabla_modelos)
tabla.loc[tabla["Modelo"] == "Polinomio de grado 3 + Lasso", "Comentario"] = \
    "el menor error · >100 columnas ilegibles · extrapola fatal"
tabla.loc[tabla["Modelo"] == "Lineal + a/c + log(edad)", "Comentario"] = \
    "9 coeficientes legibles · extrapola con sentido"
display(tabla)

tabla.to_csv("tabla_modelos_hormigon.csv", index=False)
print("Tabla guardada en tabla_modelos_hormigon.csv")

> ### 💡 Pregunta final
>
> ¿Qué modelo llevaríais a la planta? ¿Por qué no el de menor error?

---
### 📌 Para llevarse

- **Lasso = mínimos cuadrados con multa:** algunos coeficientes valen exactamente cero (selección automática). Escalar antes y elegir `alpha` por validación cruzada.
- **El Lasso doma modelos con muchas columnas:** el polinomio de grado 3 pasa de ≈ 7,7 a ≈ 6,1 MPa y es mucho más estable.
- **Menor error no es mejor modelo:** el polinomio sigue siendo ilegible y se dispara fuera del rango.
- **El criterio de ingeniero manda:** con a/c y log(edad), una regresión lineal de 9 variables se acerca mucho, se entiende y extrapola con sentido.